# Mini-FORESIGHT — Step 5: Exploratory Data Analysis (EDA)

In the previous notebooks we **understood** and **cleaned** the data.
Now we **explore** it visually to discover patterns, trends and relationships.

EDA questions we will answer:
- How does total daily sales change over time?
- Which SKU sells the most?
- How spread out are daily sales?
- Is there a weekday pattern?
- How do inventory levels evolve over time?
- Is there a relationship between stock and sales?

We will **NOT**:
- Build features
- Train a machine learning model
- Create a Streamlit app

## 1. Import Libraries

We use **pandas** for data, **matplotlib** and **seaborn** for charts.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make charts appear inside the notebook
%matplotlib inline

# Consistent, readable chart style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('pandas version:', pd.__version__)
print('matplotlib version:', plt.matplotlib.__version__)
print('seaborn version:', sns.__version__)

pandas version: 3.0.5
matplotlib version: 3.11.1
seaborn version: 0.13.2


## 2. Load the Cleaned Data

We load the cleaned files from `data/processed/` that we created in the data cleaning notebook.

In [ ]:
processed_dir = Path('data/processed')

sales_daily = pd.read_csv(processed_dir / 'sales_daily_clean.csv')
sku_master = pd.read_csv(processed_dir / 'sku_master_clean.csv')
inventory_snapshots = pd.read_csv(processed_dir / 'inventory_snapshots_clean.csv')

# Convert dates back to datetime for plotting
sales_daily['date'] = pd.to_datetime(sales_daily['date'])
inventory_snapshots['date'] = pd.to_datetime(inventory_snapshots['date'])

print('Loaded cleaned files:')
print('  sales_daily         ->', sales_daily.shape)
print('  sku_master          ->', sku_master.shape)
print('  inventory_snapshots ->', inventory_snapshots.shape)

## 3. Sales Over Time

We aggregate total units sold per day (all SKUs combined) and plot a line chart.
This shows the overall trend and any spikes or dips.

In [ ]:
daily_total = sales_daily.groupby('date')['units_sold'].sum().reset_index()

plt.figure()
sns.lineplot(data=daily_total, x='date', y='units_sold', marker='o')
plt.title('Total Daily Sales (All SKUs)')
plt.xlabel('Date')
plt.ylabel('Units Sold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Best day:', daily_total.loc[daily_total['units_sold'].idxmax(), 'date'].date(),
      '->', daily_total['units_sold'].max(), 'units')
print('Worst day:', daily_total.loc[daily_total['units_sold'].idxmin(), 'date'].date(),
      '->', daily_total['units_sold'].min(), 'units')

## 4. Total Sales by SKU

A bar chart shows which product sells the most over the whole period.

In [ ]:
total_by_sku = sales_daily.groupby('sku_id')['units_sold'].sum().reset_index()

# Add product names for a friendlier chart
total_by_sku = total_by_sku.merge(sku_master[['sku_id', 'product_name']], on='sku_id')
total_by_sku['label'] = total_by_sku['sku_id'] + ' - ' + total_by_sku['product_name']

plt.figure()
sns.barplot(data=total_by_sku, x='label', y='units_sold', hue='label', legend=False, palette='viridis')
plt.title('Total Units Sold by SKU (14 days)')
plt.xlabel('SKU')
plt.ylabel('Total Units Sold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print(total_by_sku[['sku_id', 'product_name', 'units_sold']].to_string(index=False))

## 5. Distribution of Daily Sales

A histogram shows how often each sales quantity occurs.
This tells us whether sales are consistent or vary a lot day to day.

In [ ]:
plt.figure()
sns.histplot(sales_daily['units_sold'], bins=range(1, 8), discrete=True)
plt.title('Distribution of Daily Units Sold (All SKUs)')
plt.xlabel('Units Sold per Day')
plt.ylabel('Number of Days')
plt.tight_layout()
plt.show()

print('Daily sales summary:')
print(sales_daily['units_sold'].describe())

## 6. Sales by Weekday

We extract the weekday name from the date and compare total sales across days of the week.
This reveals any weekly seasonality.

In [ ]:
sales_daily['weekday'] = sales_daily['date'].dt.day_name()

# Order weekdays Monday -> Sunday
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

weekday_sales = sales_daily.groupby('weekday')['units_sold'].sum().reindex(weekday_order).reset_index()

plt.figure()
sns.barplot(data=weekday_sales, x='weekday', y='units_sold', hue='weekday', legend=False, palette='coolwarm')
plt.title('Total Sales by Weekday (All SKUs)')
plt.xlabel('Weekday')
plt.ylabel('Total Units Sold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(weekday_sales.to_string(index=False))

## 7. Sales by Weekday per SKU

A grouped bar chart shows the weekday pattern for each SKU separately.
This helps us see if different products have different weekly rhythms.

In [ ]:
weekday_sku = sales_daily.groupby(['weekday', 'sku_id'])['units_sold'].sum().reset_index()
weekday_sku['weekday'] = pd.Categorical(weekday_sku['weekday'], categories=weekday_order, ordered=True)
weekday_sku = weekday_sku.sort_values('weekday')

plt.figure()
sns.barplot(data=weekday_sku, x='weekday', y='units_sold', hue='sku_id', palette='Set2')
plt.title('Sales by Weekday per SKU')
plt.xlabel('Weekday')
plt.ylabel('Total Units Sold')
plt.xticks(rotation=30)
plt.legend(title='SKU')
plt.tight_layout()
plt.show()

print(weekday_sku.pivot(index='weekday', columns='sku_id', values='units_sold'))

## 8. Inventory Levels Over Time

We plot closing stock per SKU over the 14 days.
This shows how stock is consumed and replenished.

In [ ]:
plt.figure()
sns.lineplot(data=inventory_snapshots, x='date', y='closing_stock', hue='sku_id', marker='o', palette='Set2')
plt.title('Closing Stock Over Time by SKU')
plt.xlabel('Date')
plt.ylabel('Closing Stock (units)')
plt.xticks(rotation=45)
plt.legend(title='SKU')
plt.tight_layout()
plt.show()

print('Minimum closing stock per SKU:')
print(inventory_snapshots.groupby('sku_id')['closing_stock'].min())

## 9. Stock vs Sales Relationship

A scatter plot of opening stock vs units sold shows whether higher stock leads to higher sales.
This is a simple way to check for a relationship between two variables.

In [ ]:
# Merge inventory and sales on (date, sku_id)
merged = inventory_snapshots.merge(
    sales_daily[['date', 'sku_id', 'units_sold']],
    on=['date', 'sku_id'],
    suffixes=('_inv', '_sales')
)

plt.figure()
sns.scatterplot(data=merged, x='opening_stock', y='units_sold_sales', hue='sku_id', palette='Set2', s=80)
plt.title('Opening Stock vs Units Sold')
plt.xlabel('Opening Stock (units)')
plt.ylabel('Units Sold')
plt.legend(title='SKU')
plt.tight_layout()
plt.show()

correlation = merged['opening_stock'].corr(merged['units_sold_sales'])
print('Correlation between opening stock and units sold: {:.3f}'.format(correlation))

## 10. Key Findings

### Sales trend
- Total daily sales (all SKUs) range from **8 to 13 units** per day.
- The best day was **2025-01-12 (13 units)** and the worst was **2025-01-02 (8 units)**.
- There is no strong upward or downward trend — sales fluctuate around **9–11 units/day**.

### Product ranking
- **SKU003 (Cushion Set)** is the best seller: **59 units** total.
- **SKU002 (Table Lamp)** is second: **50 units** total.
- **SKU001 (Wooden Chair)** sells least: **29 units** total.
- Daily sales ranges: SKU001 1–3, SKU002 2–5, SKU003 3–6.

### Distribution
- Most daily sales values are between **2 and 4 units** (median 3, mean 3.29).
- Sales are low and stable — no extreme outliers.

### Weekday pattern
- **Sunday** is the strongest day (24 units), followed by **Friday** (22) and **Saturday** (20).
- **Thursday** is the weakest day (16 units).
- The pattern is mild — the difference between best and worst weekday is only 8 units over 14 days.
- Each weekday appears exactly **2 times** in the 14-day window, so the comparison is fair.

### Inventory
- Closing stock declines for all SKUs over the period: SKU001 28→11, SKU002 27→5, SKU003 26→6.
- Minimum closing stock: SKU001 11, SKU002 5, SKU003 6 — no SKU ever runs out.
- Replenishments arrive on a few days (e.g., SKU001 on 2025-01-08, SKU002 on 2025-01-06 and 2025-01-10, SKU003 on 2025-01-05 and 2025-01-10), which creates the sawtooth pattern in the stock chart.

### Stock vs sales
- The correlation between opening stock and units sold is **-0.013**, which is essentially **zero**.
- This means stock level does **not** drive sales — demand is independent of how much inventory is on hand.

### Conclusion for forecasting
- Demand is **low, stable and roughly flat** — a simple model (e.g., using the historical average per SKU) should work well.
- There is a **mild weekend effect** (Sunday/Friday/Saturday slightly stronger) that could be used as a feature later.
- Inventory is not a constraint, so we can forecast sales without worrying about stockouts in this dataset.